# JASSIM AI — Kaggle GPU evaluation (free-session mode)

دفتر اختبار مؤقت للنموذج المفتوح على GPU المجاني المتاح في Kaggle. لا يحتاج إلى مفاتيح API أو أسرار أو حسابات خارجية مرتبطة بالمشروع.

**حدود الخصوصية:** يرسل هذا الدفتر بيانات اختبار اصطناعية فقط إلى Ollama على `127.0.0.1`. لا ينشئ نفقاً عاماً، ولا يحدّث `dashboard/connection.json`، ولا يرفع المحادثات أو قاعدة البيانات إلى GitHub/Drive. لا تستخدمه مع رسائل حقيقية.

**الاستمرارية:** جلسات Kaggle وموارد GPU المجانية مؤقتة وخاضعة للحصة والتوفر؛ هذا ليس خادماً دائماً ولا يثبت تكامل Android من طرف إلى طرف.


## 1. تحقق من بيئة GPU والاتصال


In [ ]:
import os, subprocess, sys, json, time

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No NVIDIA GPU detected in this session.')
if gpu.returncode != 0:
    raise RuntimeError('Enable a Kaggle GPU accelerator in Notebook Settings, then restart the session.')
print('GPU session detected. No secrets are required.')


## 2. احصل على كود الفرع المحدد


In [ ]:
from pathlib import Path
PROJECT = Path('/kaggle/working/JASSIM-AI-Android')
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','fix/android-message-line-breaks',
                    'https://github.com/stevennpllk-crypto/-JASSIM-AI-Android.git',str(PROJECT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin','fix/android-message-line-breaks'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'checkout','FETCH_HEAD'],check=True)
BACKEND = PROJECT / 'backend'
assert (BACKEND / 'local_model_benchmark.py').is_file()
sys.path.insert(0, str(BACKEND))
print('Checked out branch:', subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','--short','HEAD'],text=True).strip())


## 3. ثبّت Ollama وشغّله محلياً


In [ ]:
install = subprocess.run(['bash','-lc','curl -fsSL https://ollama.com/install.sh | sh'],capture_output=True,text=True)
print((install.stdout or '')[-2000:])
if install.returncode != 0:
    print((install.stderr or '')[-3000:])
    raise RuntimeError('Ollama installation failed')

ollama_log = open('/kaggle/working/ollama.log','w')
ollama_proc = subprocess.Popen(['ollama','serve'],stdout=ollama_log,stderr=subprocess.STDOUT)
import urllib.request
for attempt in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=3) as r:
            assert r.status == 200
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not become ready; inspect /kaggle/working/ollama.log')
print('Ollama is listening on loopback only.')


## 4. نزّل نموذج الاختبار المفتوح

يتم تنزيل أوزان النموذج من سجل Ollama إلى جلسة Kaggle المؤقتة. لا يتم تنزيل محادثات المستخدمين. اخترنا 4B كبداية لتقليل استهلاك الذاكرة.


In [ ]:
MODEL = 'qwen3.5:4b'
pull = subprocess.run(['ollama','pull',MODEL],text=True)
if pull.returncode != 0:
    raise RuntimeError('Model download failed; check Kaggle internet availability and GPU quota.')
print('Model weights downloaded into this temporary runtime:', MODEL)


## 5. شغّل اختبارات JASSIM الاصطناعية

الاختبارات تقيس العربية والتونسية والحساب والمنطق واتباع التعليمات واسترجاع سياق تجريبي. النتيجة ليست تقييماً شاملاً ولا مقارنة نهائية مع Gemini/ChatGPT.


In [ ]:
from local_model_benchmark import run_benchmark
import adaptive
BASE_URL = 'http://127.0.0.1:11434'
report = run_benchmark(BASE_URL, MODEL, adaptive.EVALUATION_CASES, timeout=240)
summary = {k:v for k,v in report.items() if k != 'cases'}
print(json.dumps(summary,ensure_ascii=False,indent=2))
output = Path('/kaggle/working/jassim-kaggle-benchmark.json')
output.write_text(json.dumps({'policy':'synthetic_prompts_only_loopback_ollama','report':report},ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved benchmark artifact:', output)


## 6. تحقق مستقل من أن Ollama يجيب فعلياً


In [ ]:
payload = json.dumps({'model':MODEL,'messages':[{'role':'user','content':'أجب بكلمة واحدة فقط: ما عاصمة تونس؟'}],
                      'stream':False,'options':{'temperature':0,'num_predict':32,'num_ctx':4096}}).encode('utf-8')
req = urllib.request.Request(BASE_URL+'/api/chat',data=payload,headers={'Content-Type':'application/json'},method='POST')
with urllib.request.urlopen(req,timeout=240) as response:
    live = json.loads(response.read().decode('utf-8'))
answer = live.get('message',{}).get('content','')
print('Model:',live.get('model'))
print('Synthetic prompt answer:',answer)
assert answer.strip(), 'Ollama returned an empty answer'
print('PASS: a real inference response was produced by the local Ollama process.')


## 7. ما الذي أثبتناه وما الذي لم نثبته؟

- عند نجاح الخلايا: ثبت تشغيل نموذج فعلي داخل جلسة GPU الحالية وقياسه على الاختبارات الاصطناعية.
- لم نفتح أي endpoint عام ولم نربط تطبيق Android بهذه الجلسة.
- لم نختبر تسجيل الدخول أو تاريخ المحادثات من APK.
- لا تحفظ أسراراً أو محادثات حقيقية في هذا الدفتر. نزّل ملف JSON من مخرجات الجلسة إن أردت الاحتفاظ بالنتائج.
- أوقف الجلسة بعد انتهاء القياس لتجنّب استهلاك الحصة المجانية.
